<a href="https://colab.research.google.com/github/VLCHS/FCNN_5_reactions/blob/main/FCNN_5_reactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Get Started

In [ ]:
!pip install -q torch lightning comet_ml

In [ ]:
import os
import comet_ml
import torch
import random
import math
import numpy as np
import pandas as pd
import torch.nn as nn
import lightning as L
import matplotlib.pyplot as plt

from torchmetrics import MetricCollection, MeanAbsoluteError, MeanSquaredError
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.optim.lr_scheduler import ReduceLROnPlateau

from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import CometLogger

from torch.utils.data import random_split
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.utils import resample

import ipywidgets as widgets
from IPython.display import display

def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    L.seed_everything(seed)

set_random_seed(42)

In [ ]:
device = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
comet_ml.login()

In [206]:
experiment_config = comet_ml.ExperimentConfig(name="p_pi_0")
exp = comet_ml.start(project_name="val_test_mae", experiment_config=experiment_config)
run_name = exp.name

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/chistiakova-vv19/val-test-mae/02bdef327aad49c6b8dacac3c14ec03a

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


In [207]:
params_dict = {
    'feature_scaler': StandardScaler(),
    'label_scaler': StandardScaler(),
    'batch_size': 256,
    'net_architecture': [5, 500, 500, 500, 1],
    'activation_function': nn.ReLU,
    # 'loss_func': 'RMSELoss()',
    'optim_func': torch.optim.Adam,
    'max_epochs': 200,
    'es_min_delta': 1e-05, #0.00001,
    'es_patience': 30,
    'lr': 0.001,
    'lr_factor':0.5,
    'lr_patience': 5,
    'lr_cooldown': 20,
}

# Подготовка данных

In [ ]:
from google.colab import files
uploaded = files.upload()

##$ \pi^{+} n$

In [ ]:
# pi_plus_n 00001
df_pi_plus_n = pd.read_csv('/content/clasdb_pi_plus_n.txt', delimiter='\t', header=None)
df_pi_plus_n.columns = ['Ebeam', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error', 'id']
df_pi_plus_n.loc[8314:65670, 'Ebeam'] = 5.754 # peculiarity of this dataset.
df_pi_plus_n['phi'] = df_pi_plus_n.phi.apply(lambda x: math.radians(x))
df_pi_plus_n['cos_phi'] = df_pi_plus_n['phi'].apply(lambda x: math.cos(x))
df_pi_plus_n['sin_phi'] = df_pi_plus_n['phi'].apply(lambda x: math.sin(x))

df_pi_plus_n = df_pi_plus_n.iloc[df_pi_plus_n[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
df_pi_plus_n = df_pi_plus_n[df_pi_plus_n.dsigma_dOmega <= df_pi_plus_n.dsigma_dOmega.quantile(0.96)]
df_pi_plus_n = df_pi_plus_n[df_pi_plus_n['error'] <= df_pi_plus_n["error"].quantile(0.96)]

df_pi_plus_n = df_pi_plus_n.drop('id', axis=1)
df_pi_plus_n = df_pi_plus_n.reset_index(drop=True)

#df_pi_plus_n['A'] = 1
# df_pi_plus_n['B'] = 0
# df_pi_plus_n['C'] = 0
# df_pi_plus_n['D'] = 0
# df_pi_plus_n['E'] = 1

df_pi_plus_n

##$ \pi^{0} p$

In [208]:
# pi_0_p 00010
df_pi_0_p = pd.read_csv('/content/clasdb_pi_0_p.txt', delimiter='\t', header=None)
df_pi_0_p.columns = ['Ebeam', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error', 'id']
df_pi_0_p['phi'] = df_pi_0_p.phi.apply(lambda x: math.radians(x))
df_pi_0_p['cos_phi'] = df_pi_0_p['phi'].apply(lambda x: math.cos(x))
df_pi_0_p['sin_phi'] = df_pi_0_p['phi'].apply(lambda x: math.sin(x))
df_pi_0_p['Ebeam'] = df_pi_0_p['Ebeam'].round(decimals=2)
df_pi_0_p = df_pi_0_p.replace({"Ebeam": {2.45: 2.44, 1.65: 1.64}})

df_pi_0_p = df_pi_0_p.iloc[df_pi_0_p[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
df_pi_0_p = df_pi_0_p.drop(df_pi_0_p[df_pi_0_p['dsigma_dOmega'] == 0].index)
df_pi_0_p = df_pi_0_p[df_pi_0_p['dsigma_dOmega'] <= df_pi_0_p["dsigma_dOmega"].quantile(0.96)]
df_pi_0_p = df_pi_0_p[df_pi_0_p['error'] <= df_pi_0_p["error"].quantile(0.96)]
df_pi_0_p = df_pi_0_p.drop(df_pi_0_p[df_pi_0_p['dsigma_dOmega'] <= df_pi_0_p["error"]].index)

df_pi_0_p = df_pi_0_p.drop('id', axis=1)
df_pi_0_p = df_pi_0_p.reset_index(drop=True)


# df_pi_0_p['A'] = 0
# df_pi_0_p['B'] = 1
# df_pi_0_p['C'] = 0
# df_pi_0_p['D'] = 1
# df_pi_0_p['E'] = 0

df_pi_0_p

,Ebeam,W,Q2,cos_theta,phi,dsigma_dOmega,error,cos_phi,sin_phi
0,1.64,1.1000,0.40,-0.9,2.356194,1.130000,0.909689,-0.707107,0.707107
1,1.64,1.1000,0.40,-0.9,4.974188,0.834000,0.830253,0.258819,-0.965926
2,1.64,1.1000,0.40,-0.7,0.785398,0.756000,0.682663,0.707107,0.707107
3,1.64,1.1000,0.40,-0.7,1.308997,2.340000,1.839669,0.258819,0.965926
4,1.64,1.1000,0.40,-0.7,1.832596,1.280000,0.702351,-0.258819,0.965926
...,...,...,...,...,...,...,...,...,...
67035,2.04,1.7875,0.65,0.9,4.057891,0.466615,0.078915,-0.608761,-0.793353
67036,2.04,1.7875,0.65,0.9,4.319690,0.483118,0.095376,-0.382683,-0.923880
67037,2.04,1.7875,0.65,0.9,4.581489,0.485355,0.093559,-0.130526,-0.991445
67038,2.04,1.7875,0.65,0.9,4.843289,0.528385,0.083915,0.130526,-0.991445


# Подготовка данных для нейронной сети

In [209]:
df = df_pi_0_p
#df = pd.concat([df_pi_plus_n, df_pi_0_p], ignore_index=True) #df_K_plus_Sigma_0, df_K_plus_Sigma_0_eps, df_K_plus_lambda, df_K_plus_lambda_eps, df_eta_p], ignore_index=True)
df

,Ebeam,W,Q2,cos_theta,phi,dsigma_dOmega,error,cos_phi,sin_phi
0,1.64,1.1000,0.40,-0.9,2.356194,1.130000,0.909689,-0.707107,0.707107
1,1.64,1.1000,0.40,-0.9,4.974188,0.834000,0.830253,0.258819,-0.965926
2,1.64,1.1000,0.40,-0.7,0.785398,0.756000,0.682663,0.707107,0.707107
3,1.64,1.1000,0.40,-0.7,1.308997,2.340000,1.839669,0.258819,0.965926
4,1.64,1.1000,0.40,-0.7,1.832596,1.280000,0.702351,-0.258819,0.965926
...,...,...,...,...,...,...,...,...,...
67035,2.04,1.7875,0.65,0.9,4.057891,0.466615,0.078915,-0.608761,-0.793353
67036,2.04,1.7875,0.65,0.9,4.319690,0.483118,0.095376,-0.382683,-0.923880
67037,2.04,1.7875,0.65,0.9,4.581489,0.485355,0.093559,-0.130526,-0.991445
67038,2.04,1.7875,0.65,0.9,4.843289,0.528385,0.083915,0.130526,-0.991445


In [210]:
df = df.drop(df[df['dsigma_dOmega'] == 0].index)
df = df.reset_index(drop=True)
df

,Ebeam,W,Q2,cos_theta,phi,dsigma_dOmega,error,cos_phi,sin_phi
0,1.64,1.1000,0.40,-0.9,2.356194,1.130000,0.909689,-0.707107,0.707107
1,1.64,1.1000,0.40,-0.9,4.974188,0.834000,0.830253,0.258819,-0.965926
2,1.64,1.1000,0.40,-0.7,0.785398,0.756000,0.682663,0.707107,0.707107
3,1.64,1.1000,0.40,-0.7,1.308997,2.340000,1.839669,0.258819,0.965926
4,1.64,1.1000,0.40,-0.7,1.832596,1.280000,0.702351,-0.258819,0.965926
...,...,...,...,...,...,...,...,...,...
67035,2.04,1.7875,0.65,0.9,4.057891,0.466615,0.078915,-0.608761,-0.793353
67036,2.04,1.7875,0.65,0.9,4.319690,0.483118,0.095376,-0.382683,-0.923880
67037,2.04,1.7875,0.65,0.9,4.581489,0.485355,0.093559,-0.130526,-0.991445
67038,2.04,1.7875,0.65,0.9,4.843289,0.528385,0.083915,0.130526,-0.991445


In [211]:
feature_columns = ["Ebeam", "W",	"Q2",	"cos_theta", "cos_phi"]
feature_data = df[feature_columns]
label_data = df['dsigma_dOmega']


#TRAIN TEST SPLIT
X_train, X_residual, y_train, y_residual = train_test_split(feature_data,
                                                            label_data,
                                                            test_size=0.2,   #0.66, #0.2,
                                                            random_state=42)

X_test, X_val, y_test, y_val = train_test_split(X_residual,
                                                y_residual,
                                                test_size=0.5,   #0.3, #0.5,
                                                random_state=42)
print(X_train, X_val, X_test)

scaler_feature = params_dict.get('feature_scaler')
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

#scale feature_data
columns_to_scale = list(X_train.columns)
X_train[columns_to_scale] = pd.DataFrame(scaler_feature.fit_transform(X_train[columns_to_scale]))
X_val[columns_to_scale] = pd.DataFrame(scaler_feature.transform(X_val[columns_to_scale]))
X_test[columns_to_scale] = pd.DataFrame(scaler_feature.transform(X_test[columns_to_scale]))

#scale label_data
scaler_target = params_dict.get('label_scaler')
y_train = pd.Series(scaler_target.fit_transform(y_train.to_frame())[:,0])
y_val = pd.Series(scaler_target.transform(y_val.to_frame())[:,0])
y_test = pd.Series(scaler_target.transform(y_test.to_frame())[:,0])

X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)
X_val = torch.tensor(X_val.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32)

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_dataloader = DataLoader(train_data, batch_size=params_dict.get('batch_size'), shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=params_dict.get('batch_size'), shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=params_dict.get('batch_size'), shuffle=False)

       Ebeam       W    Q2  cos_theta   cos_phi
4377    2.44  1.4000  1.45   0.100000  0.707107
65847   2.04  1.7625  0.55  -0.899999 -0.382683
65711   2.04  1.7625  0.45  -0.300007  0.991445
50693   2.04  1.4375  0.65  -0.500000  0.991445
44492   2.04  1.3125  0.75   0.100001 -0.130526
...      ...     ...   ...        ...       ...
37194   2.04  1.1625  0.75   0.500000 -0.793353
6265    2.44  1.3000  1.80  -0.700000 -0.965926
54886   2.04  1.5125  0.95  -0.700000  0.608761
860     1.64  1.3400  0.40  -0.100000  0.707107
15795   2.44  1.6000  0.65  -0.900000  0.965926

[53632 rows x 5 columns]        Ebeam       W    Q2  cos_theta   cos_phi
35859   2.04  1.1375  0.85  -0.700000 -0.991445
128     1.64  1.1200  0.40  -0.100000 -0.258819
31344   5.75  1.2300  5.00   0.300000  0.707107
17053   2.44  1.2600  0.75  -0.500000 -0.258819
13114   1.64  1.2800  0.90  -0.900000 -0.258819
...      ...     ...   ...        ...       ...
24321   2.44  1.4000  1.15  -0.100000  0.258819
18933   2.44  

# Создание модели

In [212]:
class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.net_architecture = params_dict.get('net_architecture')
        self.activation_function = params_dict.get('activation_function')

        self.network = nn.Sequential()
        for i in range(1,len(self.net_architecture)):
            self.network.append(nn.Linear(self.net_architecture[i-1], self.net_architecture[i]))
            if i!=len(self.net_architecture)-1:
                self.network.append(self.activation_function())

            else:
                pass

    def forward(self, x):
        return self.network(x)

In [213]:
model = NeuralNetwork()

# Обучение

In [214]:
class Pipeline(L.LightningModule):
    def __init__(self, model, params, scaler_target=None):
        super().__init__()

        self.model = model
        self.params = params
        self.criterion = torch.nn.MSELoss()
        self.optimizer = self.params.get('optim_func')
        self.scaler_target = scaler_target

        self.metrics = MetricCollection([
            MeanAbsoluteError(),
            MeanSquaredError(),
            #R2Score()
        ])

        self.save_hyperparameters(ignore=['model', 'scaler_target'])

        self.train_metrics = self.metrics.clone(postfix='/train')
        self.val_metrics = self.metrics.clone(postfix='/val')
        self.test_metrics = self.metrics.clone(postfix='/test')

    def configure_optimizers(self):
        optimizer = self.optimizer(self.parameters(), lr=self.params.get('lr'))

        lr_optim = ReduceLROnPlateau(optimizer = optimizer,
                                     mode = 'min',
                                     factor = self.params.get('lr_factor'),
                                     patience = self.params.get('lr_patience'),
                                     cooldown=self.params.get('lr_cooldown'),
                                     threshold=0.01
                                     )
        return {"optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": lr_optim,
                    "interval": "epoch",
                    "monitor": "val_loss",
                    "frequency": 2,
                    "name": 'lr_scheduler_monitoring'
                    },
                }

    def training_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        loss = torch.sqrt(self.criterion(out.reshape(-1), y))


        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.train_metrics.update(out_real.reshape(-1), y_real)

        else:
            self.train_metrics.update(out.reshape(-1), y)

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss


    def validation_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        loss = torch.sqrt(self.criterion(out.reshape(-1), y))

        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.val_metrics.update(out_real.reshape(-1), y_real)

        else:
            self.val_metrics.update(out.reshape(-1), y)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)


    def on_validation_epoch_end(self):
        self.log_dict(self.val_metrics.compute(), prog_bar=True)
        self.val_metrics.reset()

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), prog_bar=True)
        self.train_metrics.reset()

    def test_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)

        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.test_metrics.update(out_real.reshape(-1), y_real)
            self.test_labels.append(y_real.cpu())
            self.test_predictions.append(out_real.cpu())

        else:
            self.test_metrics.update(out.reshape(-1), y)
            self.test_labels.append(y.cpu())
            self.test_predictions.append(out.cpu())


    def on_test_start(self):
        self.test_labels = []
        self.test_predictions = []

    def on_test_epoch_end(self):
        self.log_dict(self.test_metrics.compute())
        self.test_metrics.reset()

        # all_labels = torch.cat(self.test_labels)
        # all_predictions = torch.cat(self.test_predictions)

        # self.results_df = pd.DataFrame({
        #     'true_label': all_labels.numpy().flatten(),
        #     'prediction': all_predictions.numpy().flatten()
        # })

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        out = self.model(x)
        return out

    def configure_callbacks(self):
        lr_monitor = LearningRateMonitor(logging_interval='epoch')

        early_stop_callback = EarlyStopping(monitor="val_loss", mode="min",
                                            min_delta=self.params.get('es_min_delta'),
                                            patience=self.params.get('es_patience'))

        return [lr_monitor, early_stop_callback]

In [215]:
L.seed_everything(42)

model = NeuralNetwork()
pl_model = Pipeline(model, params=params_dict, scaler_target=scaler_target)
comet_logger = CometLogger()

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    dirpath=f"./checkpoints/run_{run_name}",
    filename="best-{epoch:02d}-{val_loss:.6f}"
)

trainer = L.Trainer(
    logger=comet_logger,
    max_epochs=params_dict.get("max_epochs"),
    accelerator=device,
    callbacks=[checkpoint_callback]
)

trainer.fit(
    model=pl_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
COMET INFO: An experiment with the same configuration options is already running and will be reused.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/checkpoints/run_p_pi_0 exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ NeuralNetwork    │  504 K │ train │     0 │
│ 1 │ criterion     │ MSELoss          │      0 │ train │     0 │
│ 2 │ metrics       │ MetricCollection │      0 │ train │     0 │
│ 3 │ train_metrics │ MetricCollection │      0 │ train │     0 │
│ 4 │ val_metrics   │ MetricCollection │      0 │ train │     0 │
│ 5 │ test_metrics  │ MetricCollection │      0 │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 504 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 504 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 22                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

COMET INFO: Uploading 7 metrics, params and output messages


results WITH **scaler for targets**:

In [216]:
best_ckpt_path = checkpoint_callback.best_model_path
best_ckpt_path

'/content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt'

In [217]:
# test_data
with torch.no_grad():
    predictions = trainer.predict(
        model=pl_model,
        dataloaders=test_dataloader,
        ckpt_path=best_ckpt_path,
        weights_only=False
    )

predictions = torch.cat(predictions, dim=0).cpu().numpy()
predictions = scaler_target.inverse_transform(predictions)

y_test = y_test.reshape(-1, 1)
y_test = scaler_target.inverse_transform(y_test)

mae_test = mean_absolute_error(y_test, predictions)
mse_test = mean_squared_error(y_test, predictions)

print("Results_on_test_data")
print("Mean Absolute Error: ", mae_test)
print("Mean Squared Error: ", mse_test)

INFO:pytorch_lightning.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, LearningRateMonitor
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


COMET INFO: Uploading 6 metrics, params and output messages


Results_on_test_data
Mean Absolute Error:  0.33218231268153703
Mean Squared Error:  0.3331625625318699


In [218]:
# val_data
with torch.no_grad():
    predictions = trainer.predict(
        model=pl_model,
        dataloaders=val_dataloader,
        ckpt_path=best_ckpt_path,
        weights_only=False
    )

predictions = torch.cat(predictions, dim=0).cpu().numpy()
predictions = scaler_target.inverse_transform(predictions)

y_val = y_val.reshape(-1, 1)
y_val= scaler_target.inverse_transform(y_val)

mae_val = mean_absolute_error(y_val, predictions)
mse_val = mean_squared_error(y_val, predictions)

print("Results_on_validation_data")
print("Mean Absolute Error: ", mae_val)
print("Mean Squared Error: ", mse_val)

INFO:pytorch_lightning.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, LearningRateMonitor
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt


Output()

Results_on_validation_data
Mean Absolute Error:  0.3274963792227601
Mean Squared Error:  0.32259526838354624


In [219]:
trainer.test(dataloaders=test_dataloader, ckpt_path='best', weights_only=False)

INFO:pytorch_lightning.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, LearningRateMonitor
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  MeanAbsoluteError/test   │    0.3321823179721832     │
│   MeanSquaredError/test   │    0.3331625759601593     │
└───────────────────────────┴───────────────────────────┘

COMET INFO: Uploading 1 metrics, params and output messages


[{'MeanAbsoluteError/test': 0.3321823179721832,
  'MeanSquaredError/test': 0.3331625759601593}]

In [222]:
# Test_data
# │  MeanAbsoluteError/test   │    0.3321823179721832     │
# │   MeanSquaredError/test   │    0.3331625759601593     │

In [220]:
trainer.test(dataloaders=val_dataloader, ckpt_path='best', weights_only=False)

INFO:pytorch_lightning.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, LearningRateMonitor
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/run_p_pi_0/best-epoch=75-val_loss=0.234792.ckpt


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  MeanAbsoluteError/test   │    0.32749637961387634    │
│   MeanSquaredError/test   │    0.3225952982902527     │
└───────────────────────────┴───────────────────────────┘

COMET INFO: Uploading 2 metrics, params and output messages


[{'MeanAbsoluteError/test': 0.32749637961387634,
  'MeanSquaredError/test': 0.3225952982902527}]

In [223]:
# val_data
# │  MeanAbsoluteError/test   │    0.32749637961387634    │
# │   MeanSquaredError/test   │    0.3225952982902527     │

In [221]:
exp.log_parameters({"test_mae": mae_test, "test_mse": mse_test, "val_mae": mae_val, "val_mse": mse_val})
exp.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : p_pi_0
COMET INFO:     url                   : https://www.comet.com/chistiakova-vv19/val-test-mae/02bdef327aad49c6b8dacac3c14ec03a
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     MeanAbsoluteError/test [2]    : (0.32749637961387634, 0.3321823179721832)
COMET INFO:     MeanAbsoluteError/train [106] : (0.3017783761024475, 0.674032986164093)
COMET INFO:     MeanAbsoluteError/val [106]   : (0.32749637961387634, 0.48927193880081177)
COMET INFO:     MeanSquaredError/test [2]     : (0.3225952982902527, 0.3331625759601593)
COMET INFO:     MeanSquaredError/train [106]  : (0.2547033131122589, 1.1761670112609863)
COMET INFO:     MeanSquaredError/val 

In [ ]:
# # without scaler for targets
# trainer.test(model=pl_model, dataloaders=test_dataloader)
# results_dataframe = pl_model.results_df
# print(results_dataframe)

# num_of_params = [5000, 4500, 181, 91, 101, 5.4, 6400, 40.8, 261, 1100]
# mae_list = [0.15968162, 0.14140316, 0.14223613, 0.17740805, 0.14791107, 0.16583956, 0.15021065, 0.16220232, 0.14119283, 0.14399641]

# plt.rcParams["figure.figsize"] = (10, 7)
# plt.scatter(num_of_params, mae_list)
# plt.grid()
# plt.title("MAE vs Number of Parameters")
# plt.xlabel("Number of Net Parameters, k")
# plt.ylabel("MAE")

# plt.show()